# OMNY fare cap — exploration

Read `CLAUDE.md` first. Run `python src/seed_cache.py` once before this notebook if you
want it to work offline against the committed fixtures.

The one question that decides the project is at the bottom: does monthly-pass share vary
by station, or is it flat?

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from farecap import constants as C, fareclass, stations

pd.set_option("display.width", 140)

## Composition: the two windows side by side

In [2]:
wide = fareclass.comparison()
wide

,ridership_2024,share_pct_2024,ridership_2026,share_pct_2026
fare_class_category,,,,
OMNY - Full Fare,30363420.0,56.602,48900551.0,83.017
Metrocard - Full Fare,6671554.0,12.437,55699.0,0.095
Metrocard - Unlimited 7-Day,4181883.0,7.796,NaN,NaN
Metrocard - Unlimited 30-Day,3981746.0,7.423,34.0,0.000
OMNY - Students,2343858.0,4.369,1571206.0,2.667
Metrocard - Other,1918729.0,3.577,205082.0,0.348
Metrocard - Fair Fare,1918087.0,3.576,3.0,0.000
Metrocard - Seniors & Disability,1811049.0,3.376,4665.0,0.008
OMNY - Other,251321.0,0.469,2345608.0,3.982


Note `Metrocard - Unlimited 7-Day` is **NaN** in 2026, not 0 — the category is absent
from the schema. Don't fillna(0) here; the distinction is real.

In [3]:
shares = fareclass.headline_shares()
for k, v in shares.items():
    print(f"{k:34s} {v:,}")

total_2024                         53,643,755.0
metrocard_share_2024               38.183
unlimited_share_2024               15.22
monthly_pass_share_2024            7.42
monthly_pass_rides_2024            3,981,746.0
total_2026                         58,904,420.0
metrocard_share_2026               0.451
unlimited_share_2026               0.0
monthly_pass_share_2026            0.0
monthly_pass_rides_2026            34.0
ridership_growth_pct               9.8


## Reconciliation

The 2026 split reconciles against an independent total. The 2024 one has never been checked — open item 2.

In [4]:
for year in sorted(C.WINDOWS):
    df = fareclass.composition(year)
    print(year, "reconciles:", df.attrs["reconciles"], "| sum:", f'{df.attrs["fare_class_sum"]:,.0f}')

2024 reconciles: True | sum: 53,643,755
2026 reconciles: True | sum: 58,904,420


## Cap arithmetic

In [5]:
for k, v in fareclass.cap_arithmetic().items():
    print(f"{k:38s} {v}")

weekly_cap                             35.0
rides_to_hit_weekly_cap                11.67
monthly_equivalent_under_weekly_cap    151.67
eta_proposed_monthly_cap               138.0
gap_per_month                          13.67
gap_per_year                           164.0
caveat                                 Compares the weekly cap against ETA's proposal, not against the retired pass price, which is unverified. Do not publish a before/after dollar figure without an MTA fare schedule.


## The decision query

Needs network — the station pull was never run during scoping. A `FLAT` verdict means
there is no geography and the piece is citywide arithmetic only. Publish that either way.

In [6]:
verdict = stations.dispersion_verdict(2024)
verdict

{'year': 2024,
 'stations_ranked': 426,
 'stations_dropped_below_floor': 2,
 'median_share_pct': 7.65,
 'p25_share_pct': 6.57,
 'p75_share_pct': 9.05,
 'iqr_pp': 2.48,
 'min_share_pct': 2.71,
 'max_share_pct': 18.1,
 'verdict': 'DISPERSED — station geography is real; the map is worth building'}

In [7]:
tbl = stations.station_table(2024)
tbl[~tbl["below_volume_floor"]].head(20)

,station_complex_id,station_complex,borough,total_ridership,monthly_pass_ridership,monthly_pass_share_pct,year,below_volume_floor
0,71,8 Av (N),Brooklyn,113348.0,20516.0,18.10,2024,False
1,78,Avenue U (N),Brooklyn,40720.0,7092.0,17.42,2024,False
2,74,18 Av (N),Brooklyn,52360.0,8706.0,16.63,2024,False
3,76,Bay Pkwy (N),Brooklyn,73033.0,12066.0,16.52,2024,False
4,266,"Elmhurst Av (M,R)",Queens,115897.0,18934.0,16.34,2024,False
5,77,Kings Hwy (N),Brooklyn,61656.0,9822.0,15.93,2024,False
6,68,Bay Pkwy (D),Brooklyn,74711.0,11650.0,15.59,2024,False
7,196,Aqueduct Racetrack (A),Queens,8002.0,1201.0,15.01,2024,False
8,66,18 Av (D),Brooklyn,61447.0,9178.0,14.94,2024,False
9,69,25 Av (D),Brooklyn,42199.0,6281.0,14.88,2024,False


In [8]:
stations.borough_rollup(2024)

,borough,monthly_pass_ridership,total_ridership,monthly_pass_share_pct
3,Queens,747231.0,8120315.0,9.20
1,Brooklyn,974247.0,12019450.0,8.11
0,Bronx,275116.0,3598558.0,7.65
2,Manhattan,1979070.0,29808643.0,6.64
4,Staten Island,6082.0,96789.0,6.28
